# 02 · Manifold - principal-tree trajectory

Fit an ElPiGraph principal tree (scFates PPT) on the UMAP, root it at the
**Mixed** hub (data-driven: the principal node most weighted by hub cells),
compute pseudotime, and rename milestones by their dominant subtype. Confirms
the hub is the topological convergence point radiating to the terminal
subtypes. Deliverable: `manifold.h5ad`, `manifold.pdf`.

In [1]:
import os, sys, glob
os.environ['R_HOME'] = sys.exec_prefix + '/lib/R/'
import numpy as np, pandas as pd
import scanpy as sc, scanpy.external as sce, scFates as scf
import matplotlib.pyplot as plt
sc.settings.verbosity = 1

In [2]:
# cwd = <root>/cancer/vertebrate/CCCA/Claude/<Category>/<Dataset>
name = os.path.basename(os.getcwd())                    # dataset (folder minus 'Data_')
cat  = os.path.basename(os.path.dirname(os.getcwd()))   # Category partition
pth  = os.path.join(*([os.pardir]*6))                   # -> MesenCoder root
pth_data = os.path.join(pth, 'data')
sys.path.append(os.path.join(pth, 'cancer', 'vertebrate', 'CCCA'))
from scripts.load import load_CCCA_adata

pth_in  = os.path.join(pth_data, 'unzip', 'CCCA', 'Data_' + name)        # raw input
pth_out = os.path.join(pth_data, 'cancer', 'vertebrate', 'CCCA', 'Claude', cat, name)  # h5ads
pth_fig = os.path.join(pth, 'figures', 'cancer', 'vertebrate', 'CCCA', 'Claude', cat, name)
pth_cb  = os.path.join(pth_fig, 'colorbars')
os.makedirs(pth_out, exist_ok=True); os.makedirs(pth_cb, exist_ok=True)

In [3]:
adata = sc.read_h5ad(os.path.join(pth_out, 'processed.h5ad'))
sc.pl.umap(adata, color='celltype', size=120)

### Fit principal tree and root at the Mixed hub
`Nodes/ppt_lambda/ppt_sigma` tuned to balance over/underfitting. The root is set
data-driven to the principal node carrying the most Mixed-hub responsibility, so
pseudotime flows outward from the convergence point.

In [4]:
scf.tl.tree(adata, method='ppt', Nodes=200, ppt_lambda=2000, ppt_sigma=0.5,
            use_rep='X_umap', seed=1, plot=False, basis='umap')
R = adata.obsm['X_R']
root_node = int(np.argmax(R[(adata.obs.celltype=='Mixed').values].sum(0)))
scf.tl.root(adata, root_node)
scf.tl.pseudotime(adata, seed=1)
print('root node', root_node, '| mean pseudotime by subtype:')
print(adata.obs.groupby('celltype', observed=True)['t'].mean().round(2).to_dict())

inferring a principal tree --> parameters used 
    200 principal points, sigma = 0.5, lambda = 2000, metric = euclidean
    fitting:  20%|##        | 10/50 [00:00<00:03, 11.77it/s]
    converged
    finished (0:00:00) --> added 
    .uns['ppt'], dictionnary containing inferred tree.
    .obsm['X_R'] soft assignment of cells to principal points.
    .uns['graph']['B'] adjacency matrix of the principal points.
    .uns['graph']['F'] coordinates of principal points in representation space.
node 85 selected as a root --> added
    .uns['graph']['root'] selected root.
    .uns['graph']['pp_info'] for each PP, its distance vs root and segment assignment.
    .uns['graph']['pp_seg'] segments network information.
projecting cells onto the principal graph
    finished (0:00:01) --> added
    .obs['edge'] assigned edge.
    .obs['t'] pseudotime value.
    .obs['seg'] segment of the tree assigned.
    .obs['milestones'] milestone assigned.
    .uns['pseudotime_list'] list of cell projection from

### Rename milestones by dominant subtype (data-driven)

In [5]:
ms = adata.obs['milestones'].astype(str)
new, seen = {}, {}
for m in ms.unique():
    dom = adata.obs.loc[ms==m, 'celltype'].value_counts().idxmax()
    seen[dom] = seen.get(dom, 0) + 1
    new[m] = dom if seen[dom]==1 else f'{dom}_{seen[dom]-1}'
# de-duplicate names that recur
scf.tl.rename_milestones(adata, new=new)
scf.pl.milestones(adata, basis='umap', annotate=True, s=120)
print('milestones ->', new)

/home/caleb/miniconda3/envs/scFates/lib/python3.11/site-packages/anndata/_core/anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
milestones -> {'117': 'Atypical', '5': 'Mixed', '198': 'Basal', '85': 'Mixed_1', '0': 'Classical', '13': 'Classical_1'}


### Save manifold + embedding figure (deliverable)

In [6]:
adata.write(os.path.join(pth_out, 'manifold.h5ad'))
fig, ax = plt.subplots(1, 1, figsize=(5,5))
sc.pl.umap(adata, color='t', cmap='cividis', size=180, colorbar_loc=None,
           frameon=False, show=False, ax=ax)
sc.pl.umap(adata, color='celltype', size=180, alpha=0, legend_loc='on data',
           legend_fontoutline=1, frameon=False, show=False, ax=ax)
ax.invert_yaxis(); ax.invert_xaxis()
ax.set_title('Head and Neck Squamous Cell Carcinoma', size=13)
ax.text(.5, -.05, 'Puram et al., 2017', ha='center', transform=ax.transAxes)
fig.savefig(os.path.join(pth_fig,'manifold.pdf'), bbox_inches='tight')